In [ ]:
!pip install -q mne snntorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.6/133.6 kB 10.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


In [ ]:
# =============================================================================
# Capstone — Step C2: Full standalone script
#   - Loads chb01 data from Google Drive
#   - Re-runs your Step B encoding + windowing across MULTIPLE records
#   - Defines your Step C1 model (SeizureSNN)
#   - Trains on a real held-out-record fold (val = chb01_16, matching your
#     Stage 1 "best fold"), small epoch count, GPU-ready
#   - Done-criterion: validation loss visibly decreases
# =============================================================================

# -----------------------------------------------------------------------------
# 0. Setup
# -----------------------------------------------------------------------------


import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import mne
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import snntorch as snn

from google.colab import drive
drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/data.zip'
extract_path = '/content/chb01_data'

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

data_dir = os.path.join(extract_path, 'data')
print("Files available:", sorted(os.listdir(data_dir)))

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# -----------------------------------------------------------------------------
# 1. Config — channels, encoding params, seizure timing per record
# -----------------------------------------------------------------------------

TARGET_CHANNELS = ['F7-T7', 'T7-P7', 'F8-T8', 'T8-P8-0']  # confirm suffix matches your EDFs
N_LEVELS = 16
N_CORE_LEVELS = 12
WINDOW_SAMPLES = 2048   # 8s @ 256Hz
SFREQ = 256

REFERENCE_FILE = 'chb01_03.edf'
REFERENCE_TMIN, REFERENCE_TMAX = 0, 300  # first 5 minutes

# -----------------------------------------------------------------------------
# *** FILL THIS IN from chb01-summary.txt before running ***
# seconds, per record, matching your chb01_16 format: (start, end)
# One record can have multiple seizures -> use a list of (start, end) tuples.
# -----------------------------------------------------------------------------
SEIZURE_TIMES = {
    'chb01_16.edf': [(1015, 1066)],
    'chb01_03.edf': [(2996, 3036)],   #
    'chb01_04.edf': [(1467, 1494)],   #
    'chb01_15.edf': [(1732, 1772)],   #[cite: 1]
    'chb01_21.edf': [(327, 420)],     #[cite: 1]
    'chb01_26.edf': [(1862, 1963)],   #[cite: 1]
}

# Which records to train on, which to hold out as validation (real fold)
TRAIN_FILES = ['chb01_03.edf', 'chb01_04.edf', 'chb01_15.edf', 'chb01_21.edf', 'chb01_26.edf']
VAL_FILES = ['chb01_16.edf']

# -----------------------------------------------------------------------------
# 2. Step B — encoding + windowing (re-run per file)
# -----------------------------------------------------------------------------

def load_raw(filename, channels=TARGET_CHANNELS):
    path = os.path.join(data_dir, filename)
    raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
    return raw.copy().pick(channels)


def compute_reference_stats(ref_data_uv, n_core_levels=N_CORE_LEVELS):
    ch_min = ref_data_uv.min(axis=1)
    ch_max = ref_data_uv.max(axis=1)
    step = (ch_max - ch_min) / n_core_levels
    return ch_min, ch_max, step


def amplitude_to_level(voltage, ch_min, ch_max, step, n_levels=N_LEVELS):
    start = ch_min - 2 * step
    edges = start + step * np.arange(n_levels + 1)
    level = np.digitize(voltage, edges) - 1
    level = np.clip(level, 0, n_levels - 1)
    return level


def encode_to_spikes(data_uv, ch_min, ch_max, step, n_levels=N_LEVELS):
    n_channels, n_samples = data_uv.shape
    spike_matrix = np.zeros((n_channels * n_levels, n_samples))
    for ch_i in range(n_channels):
        levels = amplitude_to_level(data_uv[ch_i], ch_min[ch_i], ch_max[ch_i], step[ch_i], n_levels)
        for t in range(n_samples):
            row = ch_i * n_levels + levels[t]
            spike_matrix[row, t] = 1
    return spike_matrix


def make_windows(spike_matrix, sfreq, seizure_intervals,
                  window_size=WINDOW_SAMPLES, label_threshold=0.5):
    n_rows, n_samples = spike_matrix.shape
    windows, labels = [], []

    for start in range(0, n_samples - window_size + 1, window_size):
        end = start + window_size
        window = spike_matrix[:, start:end]

        overlap = 0
        for (sz_start, sz_end) in seizure_intervals:
            sz_start_sample = sz_start * sfreq
            sz_end_sample = sz_end * sfreq
            overlap += max(0, min(end, sz_end_sample) - max(start, sz_start_sample))
        fraction = overlap / window_size
        label = 1 if fraction >= label_threshold else 0

        windows.append(window)
        labels.append(label)

    return np.array(windows), np.array(labels)


def process_record(filename, ch_min, ch_max, step):
    """Load one EDF, encode to spikes, window + label it."""
    raw = load_raw(filename)
    data_uv = raw.get_data() * 1e6
    spike_matrix = encode_to_spikes(data_uv, ch_min, ch_max, step)
    seizure_intervals = SEIZURE_TIMES.get(filename, [])
    windows, labels = make_windows(spike_matrix, SFREQ, seizure_intervals)
    print(f"{filename}: {windows.shape[0]} windows, {int(labels.sum())} seizure-labeled")
    return windows, labels


# -----------------------------------------------------------------------------
# 3. Compute reference stats once (from chb01_03, first 5 min), reuse for all files
# -----------------------------------------------------------------------------

raw_ref = load_raw(REFERENCE_FILE)
ref_data_uv = raw_ref.get_data(tmin=REFERENCE_TMIN, tmax=REFERENCE_TMAX) * 1e6
ch_min, ch_max, step = compute_reference_stats(ref_data_uv)
print("Reference stats — ch_min:", ch_min, "ch_max:", ch_max, "step:", step)

# -----------------------------------------------------------------------------
# 4. Build train set (concatenate multiple records) and val set (chb01_16)
# -----------------------------------------------------------------------------

train_windows_list, train_labels_list = [], []
for f in TRAIN_FILES:
    if len(SEIZURE_TIMES.get(f, [])) == 0:
        print(f"WARNING: no seizure times set for {f} — check SEIZURE_TIMES before trusting labels")
    w, l = process_record(f, ch_min, ch_max, step)
    train_windows_list.append(w)
    train_labels_list.append(l)

train_windows = np.concatenate(train_windows_list, axis=0)
train_labels = np.concatenate(train_labels_list, axis=0)

val_windows_list, val_labels_list = [], []
for f in VAL_FILES:
    w, l = process_record(f, ch_min, ch_max, step)
    val_windows_list.append(w)
    val_labels_list.append(l)

val_windows = np.concatenate(val_windows_list, axis=0)
val_labels = np.concatenate(val_labels_list, axis=0)

print(f"\nTrain: {train_windows.shape[0]} windows, {int(train_labels.sum())} seizure")
print(f"Val:   {val_windows.shape[0]} windows, {int(val_labels.sum())} seizure")

# -----------------------------------------------------------------------------
# 5. Step C1 — model
# -----------------------------------------------------------------------------

class SeizureSNN(nn.Module):
    """
    64 -> 8 -> 1 LIF SNN
    fc1: 64*8 + 8 = 520
    fc2: 8*1 + 1  = 9
    Total = 529
    beta and threshold fixed (not learnable).
    """
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size, bias=True)
        self.fc2 = nn.Linear(hidden_size, output_size, bias=True)
        self.lif1 = snn.Leaky(beta=0.9, threshold=1.0, learn_beta=False,
                               learn_threshold=False, reset_mechanism="subtract")
        self.lif2 = snn.Leaky(beta=0.9, threshold=1.0, learn_beta=False,
                               learn_threshold=False, reset_mechanism="subtract")

    def forward(self, x):
        # x: (num_steps, batch_size, 64)

        # Vectorize fc1 across all timesteps outside the loop
        cur1_all = self.fc1(x)  # Shape becomes (num_steps, batch_size, hidden_size)

        # Initialize membrane potentials
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()

        spk_rec, mem_rec = [], []
        num_steps = x.size(0)

        for step_i in range(num_steps):
            # Pass the pre-computed current for this timestep
            spk1, mem1 = self.lif1(cur1_all[step_i], mem1)

            # fc2 still requires the recurrent spikes from lif1
            cur2 = self.fc2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)

            spk_rec.append(spk2)
            mem_rec.append(mem2)

        return torch.stack(spk_rec), torch.stack(mem_rec)


# Sanity check (matches your Step C1 checks)
_model_check = SeizureSNN(64, 8, 1)
_total_params = sum(p.numel() for p in _model_check.parameters())
print(f"Total learnable params: {_total_params} (target: 529)")
del _model_check

# -----------------------------------------------------------------------------
# 6. Dataset / DataLoader — handles Step B's (n, 64, 2048) -> model's (T, B, 64)
# -----------------------------------------------------------------------------

class SpikeWindowDataset(Dataset):
    def __init__(self, windows, labels):
        # (n_windows, 64, 2048) -> (n_windows, 2048, 64)
        self.windows = torch.from_numpy(windows).float().permute(0, 2, 1)
        self.labels = torch.from_numpy(labels).float()

    def __len__(self):
        return self.windows.shape[0]

    def __getitem__(self, idx):
        return self.windows[idx], self.labels[idx]


def collate_time_major(batch):
    x = torch.stack([item[0] for item in batch])  # (batch, 2048, 64)
    y = torch.stack([item[1] for item in batch])  # (batch,)
    return x.permute(1, 0, 2), y  # -> (2048, batch, 64)


BATCH_SIZE = 32

train_ds = SpikeWindowDataset(train_windows, train_labels)
val_ds = SpikeWindowDataset(val_windows, val_labels)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_time_major,
    num_workers=2,         # Offloads batch preparation to subprocesses
    pin_memory=True        # Stages tensors in page-locked memory for fast GPU transfer
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_time_major,
    num_workers=2,
    pin_memory=True
)
# -----------------------------------------------------------------------------
# 7. Loss — firing-rate MSE (matches Paper #7 / your old notebook)
# -----------------------------------------------------------------------------

TARGET_RATE_NORMAL = 0.03
TARGET_RATE_SEIZURE = 0.35

def firing_rate_loss(spk_rec, labels):
    mean_rate = spk_rec.mean(dim=0).squeeze(-1)  # (batch,)
    target = torch.where(labels == 1,
                          torch.tensor(TARGET_RATE_SEIZURE, device=labels.device),
                          torch.tensor(TARGET_RATE_NORMAL, device=labels.device))
    return F.mse_loss(mean_rate, target), mean_rate

# -----------------------------------------------------------------------------
# 8. Step C2 — training loop
# -----------------------------------------------------------------------------

def train_one_fold(model, train_loader, val_loader, num_epochs=10, lr=1e-3, device="cpu"):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(num_epochs):
        model.train()
        running_loss, n_batches = 0.0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            spk_rec, _ = model(x)
            loss, _ = firing_rate_loss(spk_rec, y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            n_batches += 1
        train_loss = running_loss / max(n_batches, 1)

        model.eval()
        running_val_loss, n_val_batches = 0.0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                spk_rec, _ = model(x)
                loss, _ = firing_rate_loss(spk_rec, y)
                running_val_loss += loss.item()
                n_val_batches += 1
        val_loss = running_val_loss / max(n_val_batches, 1)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        print(f"Epoch {epoch+1:2d}/{num_epochs} | train_loss: {train_loss:.5f} | val_loss: {val_loss:.5f}")

    return history


model = SeizureSNN(64, 8, 1)
history = train_one_fold(model, train_loader, val_loader, num_epochs=10, lr=1e-3, device=device)

print(f"\nVal loss: {history['val_loss'][0]:.5f} -> {history['val_loss'][-1]:.5f}")

# Save results so you have them after your break
torch.save(model.state_dict(), '/content/drive/MyDrive/c2_model_chb01_16_val.pt')
np.save('/content/drive/MyDrive/c2_history.npy', history)
print("Saved model + history to Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files available: ['.DS_Store', 'chb01-summary.txt', 'chb01_01.edf', 'chb01_03.edf', 'chb01_03.edf.seizures', 'chb01_04.edf', 'chb01_04.edf.seizures', 'chb01_05.edf', 'chb01_15.edf', 'chb01_15.edf.seizures', 'chb01_16.edf', 'chb01_16.edf.seizures', 'chb01_18.edf', 'chb01_18.edf.seizures', 'chb01_21.edf', 'chb01_21.edf.seizures', 'chb01_26.edf', 'chb01_26.edf.seizures']
Using device: cuda


/tmp/ipykernel_3148/2290169435.py:79: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(path, preload=True, verbose=False)


Reference stats — ch_min: [-232.67399267 -244.78632479 -198.29059829 -165.86080586] ch_max: [181.48962149 154.13919414 192.42979243 170.94017094] step: [34.51363451 33.24379324 32.56003256 28.06674807]


/tmp/ipykernel_3148/2290169435.py:79: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(path, preload=True, verbose=False)


chb01_03.edf: 450 windows, 6 seizure-labeled


...

chb01_21.edf: 450 windows, 12 seizure-labeled


/tmp/ipykernel_3148/2290169435.py:79: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(path, preload=True, verbose=False)


chb01_26.edf: 290 windows, 12 seizure-labeled


/tmp/ipykernel_3148/2290169435.py:79: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(path, preload=True, verbose=False)


chb01_16.edf: 450 windows, 6 seizure-labeled

Train: 2090 windows, 40 seizure
Val:   450 windows, 6 seizure
Total learnable params: 529 (target: 529)
Epoch  1/10 | train_loss: 0.00275 | val_loss: 0.00157
Epoch  2/10 | train_loss: 0.00155 | val_loss: 0.00145
Epoch  3/10 | train_loss: 0.00119 | val_loss: 0.00094
Epoch  4/10 | train_loss: 0.00095 | val_loss: 0.00076
Epoch  5/10 | train_loss: 0.00079 | val_loss: 0.00064
Epoch  6/10 | train_loss: 0.00076 | val_loss: 0.00069
Epoch  7/10 | train_loss: 0.00066 | val_loss: 0.00054
Epoch  8/10 | train_loss: 0.00063 | val_loss: 0.00056
Epoch  9/10 | train_loss: 0.00062 | val_loss: 0.00052
Epoch 10/10 | train_loss: 0.00062 | val_loss: 0.00051

Val loss: 0.00157 -> 0.00051
Saved model + history to Drive.


In [ ]:
from sklearn.metrics import (accuracy_score, confusion_matrix,
                              roc_auc_score, roc_curve)

from google.colab import drive
drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/data.zip'
extract_path = '/content/chb01_data'
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)
data_dir = os.path.join(extract_path, 'data')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


model = SeizureSNN(64, 8, 1)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.to(device)
model.eval()
print("Loaded model from", MODEL_PATH)

# -----------------------------------------------------------------------------
# Inference — get firing rate per val window
# -----------------------------------------------------------------------------

x = torch.from_numpy(val_windows).float().permute(2, 0, 1).to(device)  # (2048, n_windows, 64)
y_true = val_labels

with torch.no_grad():
    spk_rec, _ = model(x)              # (2048, n_windows, 1)
    mean_rate = spk_rec.mean(dim=0).squeeze(-1).cpu().numpy()  # (n_windows,) firing rate per window

print(f"\nFiring rate — normal windows: mean {mean_rate[y_true==0].mean():.4f}")
if (y_true == 1).sum() > 0:
    print(f"Firing rate — seizure windows: mean {mean_rate[y_true==1].mean():.4f}")

# -----------------------------------------------------------------------------
# Metrics
# -----------------------------------------------------------------------------

# AUC uses the continuous firing rate directly — no threshold needed
auc = roc_auc_score(y_true, mean_rate)
print(f"\nAUC: {auc:.4f}")

# For accuracy/sensitivity/specificity we need a threshold. Two honest options:
# (1) fixed midpoint between target rates (0.03, 0.35) -> 0.19
# (2) Youden's-J optimal threshold from the ROC curve (what your old notebook
#     switched to after the fixed threshold gave 0% sensitivity)
FIXED_THRESHOLD = (0.03 + 0.35) / 2  # 0.19

fpr, tpr, thresholds = roc_curve(y_true, mean_rate)
youden_j = tpr - fpr
best_idx = np.argmax(youden_j)
YOUDEN_THRESHOLD = thresholds[best_idx]

def compute_metrics(y_true, scores, threshold, label):
    y_pred = (scores >= threshold).astype(int)
    acc = accuracy_score(y_true, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
    specificity = tn / (tn + fp) if (tn + fp) > 0 else float('nan')
    print(f"\n--- {label} (threshold={threshold:.4f}) ---")
    print(f"Accuracy:    {acc:.4f}")
    print(f"Sensitivity: {sensitivity:.4f}  (recall on seizure windows)")
    print(f"Specificity: {specificity:.4f}  (recall on normal windows)")
    print(f"Confusion matrix: TN={tn} FP={fp} FN={fn} TP={tp}")
    return acc, sensitivity, specificity

compute_metrics(y_true, mean_rate, FIXED_THRESHOLD, "Fixed threshold (0.19 midpoint)")
compute_metrics(y_true, mean_rate, YOUDEN_THRESHOLD, "Youden's-J threshold (ROC-optimal)")

print("""
NOTE for presentation: this model is from Step C2 — 10 epochs, single
held-out-record fold, per-timestep loop (not yet vectorized). It's a
bug-catching checkpoint, not the final result. Step C3 fixes the sequential
bottleneck and runs the full 7-fold sweep with a realistic epoch count,
targeting the old AUC ~0.998 for comparison.
""")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.6/133.6 kB 12.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
Mounted at /content/drive
Using device: cuda


/tmp/ipykernel_1121/3350769111.py:60: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
/tmp/ipykernel_1121/3350769111.py:60: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(path, preload=True, verbose=False)


Val: 450 windows, 6 seizure-labeled
Loaded model from /content/drive/MyDrive/c2_model_chb01_16_val.pt

Firing rate — normal windows: mean 0.0316
Firing rate — seizure windows: mean 0.2688

AUC: 1.0000

--- Fixed threshold (0.19 midpoint) (threshold=0.1900) ---
Accuracy:    0.9978
Sensitivity: 0.8333  (recall on seizure windows)
Specificity: 1.0000  (recall on normal windows)
Confusion matrix: TN=444 FP=0 FN=1 TP=5

--- Youden's-J threshold (ROC-optimal) (threshold=0.1602) ---
Accuracy:    1.0000
Sensitivity: 1.0000  (recall on seizure windows)
Specificity: 1.0000  (recall on normal windows)
Confusion matrix: TN=444 FP=0 FN=0 TP=6

NOTE for presentation: this model is from Step C2 — 10 epochs, single
held-out-record fold, per-timestep loop (not yet vectorized). It's a
bug-catching checkpoint, not the final result. Step C3 fixes the sequential
bottleneck and runs the full 7-fold sweep with a realistic epoch count,
targeting the old AUC ~0.998 for comparison.

